# 🧪 PT-W4-D2 概念实验：State / Event / Effect 三层分离，让 Agent 能回答"为什么"

> 配套阅读：`PT-W4-D2-升级Lifecycle与Event.md`（守卫显式化、命名规范、治理分层在那边）
> 这个 notebook 把第四节的**声明表**写成可执行代码，然后跑第五节的 **A101 场景 L2 验证**。

```
State（状态机节点） --迁移--> Event（不可变事实） --触发--> Effect（对其他类型对象的影响）
```

## 第 1 格：类型层（冻结）—— effect-registry.yaml 的 5 类

registry 只回答"世界上**存在**哪些影响类型"。这一层是冻结的，加规则不开 registry。

In [ ]:
# 类型层：等价于 effect-registry.yaml v1.0（5 类冻结）
EFFECT_REGISTRY = {
    "occupancy":       {"target_object_type": "ResourceUnit", "owner_domain": "Lease/Occupancy"},
    "financial":       {"target_object_type": "BillAR",        "owner_domain": "Billing & AR"},
    "state-transition":{"target_object_type": "*",             "owner_domain": "各域自管"},
    "lead-conversion": {"target_object_type": "Lead",          "owner_domain": "招商"},
    "maintenance":     {"target_object_type": "OperationTask", "owner_domain": "运营"},
}

print("已注册 effect 类型：")
for k, v in EFFECT_REGISTRY.items():
    print(f"  {k:<16} → target_object_type: {v['target_object_type']:<14} owner: {v['owner_domain']}")

## 第 2 格：规则层（声明表）—— Contract 状态机的迁移声明

每个迁移 = 守卫 + 过去式 Event + Effects。**Effect 只声明 target_object_type，不声明 id**（实例解析交给运行时，沿 D1 的 `occupies` 关系走）。

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class TransitionDecl:
    from_state: str
    to_state: str
    guard: str                       # 守卫：什么条件下允许迁移（显式化！）
    event: str                       # 命名即语义：过去式 = 事实，不是动作
    effects: tuple                   # [(effect_type,)] 规则层不绑实例 id

CONTRACT_TRANSITIONS = {
    ("Draft", "Approved"):     TransitionDecl("Draft", "Approved", "审批通过",
                                              "ContractApproved",  ("occupancy",)),
    ("Active", "Expired"):     TransitionDecl("Active", "Expired", "租期到期日到达",
                                              "ContractExpired",   ("financial",)),
    ("Active", "Terminated"):  TransitionDecl("Active", "Terminated",
                                              "终止审批通过 ∧ 退租Inspection完成",
                                              "ContractTerminated", ("occupancy", "financial")),
}

CONTRACT_STATES = ["Draft", "Approved", "Active", "Expiring", "Expired", "Terminated"]

for (f, t), d in CONTRACT_TRANSITIONS.items():
    print(f"{f:>8} → {t:<10} 守卫[{d.guard}]")
    print(f"{'':>11}Event: {d.event}  Effects: {d.effects}")

## 第 3 格：实例层 —— A101 的世界（D1 的 Relationship 在这里用上）

`occupies` 关系是 D1 建立的。Effect 规则不写"A101"，运行时沿关系解析出 A101。

In [ ]:
@dataclass
class Instance:
    obj_type: str
    obj_id: str
    state: str

# 实例世界：A101 铺位 + 一份进行中的终止合同
resource_unit = Instance("ResourceUnit", "A101", state="使用中")
contract      = Instance("Contract", "C-2026-0810", state="Active")

# D1 层建立的关系（occupies）
relationships = {("occupies", "C-2026-0810"): "A101"}

# 守卫的外部事实（ BCM 域文件里的前置条件）
facts = {
    "终止审批通过": True,
    "退租Inspection完成": False,   # ← 还没验房！
}

print(f"ResourceUnit {resource_unit.obj_id}: state={resource_unit.state}")
print(f"Contract {contract.obj_id}: state={contract.state}")
print(f"守卫事实: {facts}")

## 第 4 格：迁移引擎 —— 守卫不满足时，被挡住的是**事件**，不是状态字段

In [ ]:
def try_transition(inst: Instance, to_state: str, decls, facts, event_log, effect_log):
    decl = decls.get((inst.state, to_state))
    if decl is None:
        return f"❌ 非法迁移 {inst.state}→{to_state}（状态机里没有这条边）"
    # 逐条检查守卫
    for condition in decl.guard.split(" ∧ "):
        if not facts.get(condition, False):
            return (f"⏸️ 守卫未满足：[{condition}=False]，"
                    f"Event `{decl.event}` 尚未发生")
    # 守卫全过 → 事实发生（不可变，追加式 event log）
    inst.state = to_state
    event_log.append((inst.obj_id, decl.event))
    # Effect 沿关系解析实例
    for eff in decl.effects:
        target = EFFECT_REGISTRY[eff]["target_object_type"]
        resolved = relationships.get(("occupies", inst.obj_id), inst.obj_id)
        effect_log.append((decl.event, eff, f"{target}:{resolved}"))
    return f"✅ {decl.event} 已发生"

event_log, effect_log = [], []
msg = try_transition(contract, "Terminated", CONTRACT_TRANSITIONS, facts, event_log, effect_log)
print("第一次尝试终止：", msg)
print()
print(f"event_log  = {event_log}   ← 空：没有事实发生")
print(f"contract.state 仍是 {contract.state!r} —— 但注意：失败的不是 status 字段，是守卫")

## 第 5 格：A101 场景 L2 验证 —— Agent 用事件回答"为什么"

验证问题：**"A101 铺位为什么不能出租？"**

- 升级前：Agent 只能说 "asset.status='使用中'，代码里这么写的"
- 升级后：Agent 沿声明表推理，引用的是**业务事实**

In [ ]:
def answer_why_not_leasable(unit_id: str) -> str:
    # L1 定位：沿关系找到关联合同
    related = [cid for (rel, cid), rid in relationships.items()
               if rel == "occupies" and rid == unit_id]
    lines = [f"查询：{unit_id} 为什么不能出租？", ""]
    for cid in related:
        # L2 业务链推理：检查释放迁移（Active→Terminated）的守卫卡在哪
        d = CONTRACT_TRANSITIONS[("Active", "Terminated")]
        unmet = [cond for cond in d.guard.split(" ∧ ") if not facts.get(cond, False)]
        if unmet:
            lines += [
                f"推理链：{unit_id} --occupies--> Contract {cid}（state={contract.state}）",
                f"释放铺位需要迁移 Active→Terminated，其守卫：{d.guard}",
                f"未满足：{unmet[0]}",
                "",
                f"回答：合同 {cid} 处于终止流程中，`{d.event}` 事件尚未发出，",
                f"occupancy-effect 未触发，铺位仍被占用，故不可出租。",
            ]
    return "\n".join(lines)

print(answer_why_not_leasable("A101"))

## 第 6 格：守卫满足后 —— 事件落地，Effect 沿关系解析

注意 Effect 日志里 target 是**运行时沿 `occupies` 解析出的 A101**，而声明表里从未出现"A101"三个字。**规则层与实例层分离**。

In [ ]:
# 验房完成了
facts["退租Inspection完成"] = True

msg = try_transition(contract, "Terminated", CONTRACT_TRANSITIONS, facts, event_log, effect_log)
print(msg)
print()
print("Event Log（不可变事实，Agent 解释'为什么'时引用这里）：")
for cid, ev in event_log:
    print(f"  {cid}: {ev}")
print()
print("Effect Log（运行时沿 occupies 关系解析实例）：")
for ev, eff, target in effect_log:
    print(f"  {ev} --[{eff}]--> {target}")

## 结论：三层分离给 AI 的可推理性

| 层 | 本实验对应 | 治理方式 |
|---|---|---|
| 类型层 | `EFFECT_REGISTRY`（5 类） | registry 冻结 |
| 规则层 | `CONTRACT_TRANSITIONS`（守卫+Event+Effects） | 声明表，随域演进 |
| 实例层 | `Instance` + `relationships` + event_log | 运行时解析 |

**Agent 的推理单位是事实（Event）不是快照（status 字段）。** 状态查完就过期，事件是因果链上的节点：可回溯、可解释。

> 练习（md 版第七题）：给 OperationTask 的工单迁移写一条声明，注意 maintenance-effect 的 owner_domain 是"运营"——费用归集该发 financial-effect 吗？跑一下 `EFFECT_REGISTRY` 再回答。

→ 深入阅读：同目录 `.md` 版本第二~五节（现状盘点 + 守卫显式化三个升级点 + ADR-006 对应）